<a href="https://colab.research.google.com/github/anuvishalp/Python_Projects/blob/main/Python-PracticalSkill1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
python skills - enough to automate something, call an API, process JSON, connect to a database, build a small AI-assisted tool, understand RAG, try tool calling, and see how an AI workflow actually works.
Try to build some Applied AI and AI Platform knowledge.

In [ ]:
Python for Applied AI: a hands-on path

Work through this in order. Each stage builds on the last,
and by the end you'll have written a small version of everything an AI platform does.

Stage 0: Setup
bash
python -m venv .venv && source .venv/bin/activate   # Windows: .venv\Scripts\activate

pip install requests pydantic python-dotenv anthropic sentence-transformers numpy fastapi uvicorn
=====================

Put secrets in a .env file (ANTHROPIC_API_KEY=...) and never in code.
Load them with from dotenv import load_dotenv; load_dotenv().

In [ ]:
Core Python you need before anything else:

functions, dicts and lists, comprehensions, f-strings,
try/except, with blocks, imports, and type hints.

========================================================
Stage 1: Automation (files and scripts)

Find text files in a folder, summarize them, and write a CSV report.

python

            from pathlib import Path
            import csv

            def scan(folder: str) -> list[dict]:
                rows = []
                for p in Path(folder).rglob("*.txt"):
                    text = p.read_text(encoding="utf-8")
                    rows.append({"file": p.name, "words": len(text.split()), "chars": len(text)})
                return rows

            if __name__ == "__main__":
                rows = scan("./notes")
                with open("report.csv", "w", newline="") as f:
                    w = csv.DictWriter(f, fieldnames=["file", "words", "chars"])
                    w.writeheader()
                    w.writerows(rows)

Skills to pick up here: pathlib, csv, argparse for command-line options,
and if __name__ == "__main__". Later you can schedule scripts with cron or Task Scheduler.


In [ ]:
Stage 2: Calling APIs and processing JSON

python
            import requests, time

            def get_json(url, params=None, retries=3):
                for attempt in range(retries):
                    try:
                        r = requests.get(url, params=params, timeout=10)
                        r.raise_for_status()
                        return r.json()
                    except requests.RequestException:
                        if attempt == retries - 1:
                            raise
                        time.sleep(2 ** attempt)   # exponential backoff

            data = get_json("https://api.github.com/repos/python/cpython")
            print(data["stargazers_count"], data.get("license", {}).get("name"))

======================
The JSON habits that matter:

json.loads() turns a string into a dict, and json.dumps(obj, indent=2) goes the other way.
Use .get("key", default) for fields that might be missing.
Always set a timeout, check the status code, and retry with backoff. Real APIs fail constantly.
==============================

Validate data at the boundary with Pydantic instead of trusting it:
python

            from pydantic import BaseModel

            class Repo(BaseModel):
                name: str
                stargazers_count: int

            repo = Repo(**data)   # raises a clear error if the shape is wrong


In [ ]:
Stage 3: Databases

SQLite ships with Python, so start there. The SQL and the patterns transfer directly to Postgres.

python
          import sqlite3

          con = sqlite3.connect("app.db")
          con.execute("""CREATE TABLE IF NOT EXISTS tickets(
              id INTEGER PRIMARY KEY, body TEXT, category TEXT, urgency INTEGER)""")

          con.execute("INSERT INTO tickets(body, category, urgency) VALUES (?, ?, ?)",
                      ("Can't log in", "auth", 4))   # ? placeholders prevent SQL injection
          con.commit()

          for row in con.execute("SELECT category, COUNT(*) FROM tickets GROUP BY category"):
              print(row)

==================================
Always use ? placeholders and never build SQL with f-strings.
Once this is comfortable, move to Postgres with psycopg and
try the pgvector extension, which is where many production RAG systems store embeddings.


In [ ]:
Stage 4: Your first LLM call

python

          import anthropic
          from dotenv import load_dotenv
          load_dotenv()

          client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY

          msg = client.messages.create(
              model="claude-sonnet-5",
              max_tokens=500,
              system="You are a concise technical assistant.",
              messages=[{"role": "user", "content": "Explain idempotency in one paragraph."}],
          )
          print(msg.content[0].text)
          print(msg.usage.input_tokens, msg.usage.output_tokens)   # this is your cost
===============================
The concepts to internalize here:

Messages alternate user and assistant. The API is stateless, so you resend the history each call.
System prompt sets role and rules.
Tokens drive both cost and the context limit.
Temperature controls randomness. Use low values for extraction, higher for creative work.


In [ ]:
Stage 5: A small AI-assisted tool (ticket triage)

This combines Stages 1 to 4: take messy text, get structured JSON from the model, validate it, and store it.

python
            import json, sqlite3, anthropic
            from pydantic import BaseModel, Field

            client = anthropic.Anthropic()

            class Triage(BaseModel):
                category: str = Field(pattern="^(auth|billing|bug|other)$")
                urgency: int = Field(ge=1, le=5)
                summary: str

            SYSTEM = """Classify support tickets. Respond with ONLY a JSON object:
            {"category": "auth|billing|bug|other", "urgency": 1-5, "summary": "<one sentence>"}"""

            def triage(body: str) -> Triage:
                for _ in range(2):   # one retry if the output is malformed
                    msg = client.messages.create(
                        model="claude-sonnet-5", max_tokens=300, system=SYSTEM,
                        messages=[{"role": "user", "content": body}])
                    text = msg.content[0].text.strip().removeprefix("```json").removesuffix("```").strip()
                    try:
                        return Triage(**json.loads(text))
                    except Exception:
                        continue
                raise ValueError("Model returned invalid output twice")

            con = sqlite3.connect("app.db")
            t = triage("I was charged twice this month and support hasn't replied in 3 days!!")
            con.execute("INSERT INTO tickets(body, category, urgency) VALUES (?,?,?)",
                        ("double charge", t.category, t.urgency))
            con.commit()
            print(t)

The lesson is that an LLM output is untrusted input. Validate it, retry it, and handle failure, exactly as you would with any external API.

In [ ]:
Stage 6: RAG (retrieval-augmented generation)

The problem: the model doesn't know your documents, and pasting all of them into the prompt is expensive and hits context limits.
The idea: find the few relevant chunks, put only those in the prompt, and ask the model to answer from them.
==================================================================================================
question → embed → search vector store → top-k chunks → prompt (chunks + question) → LLM → answer
==================================================================================================
A complete version from scratch:

python

      import numpy as np, anthropic
      from sentence_transformers import SentenceTransformer

      embedder = SentenceTransformer("all-MiniLM-L6-v2")   # runs locally, free
      client = anthropic.Anthropic()

      def chunk(text, size=400, overlap=50):
          words = text.split()
          step = size - overlap
          return [" ".join(words[i:i + size]) for i in range(0, len(words), step)]

# 1. Index: chunk documents and embed them
      docs = {"refunds.txt": open("refunds.txt").read(), "shipping.txt": open("shipping.txt").read()}
      chunks, sources = [], []
      for name, text in docs.items():
          for c in chunk(text):
              chunks.append(c); sources.append(name)
      vectors = embedder.encode(chunks, normalize_embeddings=True)

# 2. Retrieve: cosine similarity (dot product on normalized vectors)
      def retrieve(question, k=3):
          q = embedder.encode([question], normalize_embeddings=True)[0]
          top = np.argsort(-(vectors @ q))[:k]
          return [(chunks[i], sources[i]) for i in top]

# 3. Generate: answer only from the retrieved context
      def ask(question):
          hits = retrieve(question)
          context = "\n\n".join(f"[{src}]\n{txt}" for txt, src in hits)
          msg = client.messages.create(
              model="claude-sonnet-5", max_tokens=600,
              system="Answer using ONLY the provided context. Cite the [source]. "
                    "If the answer isn't in the context, say you don't know.",
              messages=[{"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}])
          return msg.content[0].text

      print(ask("What's the refund window for damaged items?"))
=======================================
What to understand about RAG:

Embeddings turn text into vectors, so similar meaning means nearby vectors.
Chunking strategy often matters more than the model. Bad chunks mean bad retrieval, which means bad answers.
Most RAG failures are retrieval failures. When an answer is wrong, print what was retrieved before blaming the LLM.
Upgrades to explore: hybrid search (keyword plus vector), reranking, metadata filters, and a vector database (pgvector, Chroma, Qdrant) instead of a NumPy array.

In [ ]:
Stage 7: Tool calling

Tool calling lets the model request that your code do something, such as query a database or hit an API. The model never runs anything itself. It emits a structured request, your code executes it, and you send the result back.

python
                import json, anthropic
                client = anthropic.Anthropic()

                def get_order_status(order_id: str) -> dict:
                    fake_db = {"A123": {"status": "shipped", "eta": "Friday"}}
                    return fake_db.get(order_id, {"error": "order not found"})

                TOOL_FUNCS = {"get_order_status": get_order_status}

                tools = [{
                    "name": "get_order_status",
                    "description": "Look up the shipping status of an order by its ID.",
                    "input_schema": {
                        "type": "object",
                        "properties": {"order_id": {"type": "string", "description": "e.g. A123"}},
                        "required": ["order_id"],
                    },
                }]

                messages = [{"role": "user", "content": "Where is order A123?"}]

                while True:
                    resp = client.messages.create(model="claude-sonnet-5", max_tokens=1000,
                                                  tools=tools, messages=messages)
                    messages.append({"role": "assistant", "content": resp.content})
                    if resp.stop_reason != "tool_use":
                        break
                    results = []
                    for block in resp.content:
                        if block.type == "tool_use":
                            out = TOOL_FUNCS[block.name](**block.input)
                            results.append({"type": "tool_result", "tool_use_id": block.id,
                                            "content": json.dumps(out)})
                    messages.append({"role": "user", "content": results})

                print("".join(b.text for b in resp.content if b.type == "text"))
================================================================================================================================
Those ~20 lines are the core of every AI agent. Try this next:

Add a second tool (for example search_docs, which wraps your RAG retrieve) and watch the model choose between them.
Add a max-iterations guard to the loop so it can't run forever.
Log every tool call, because you'll need that when debugging.

In [ ]:
Stage 8: How an AI workflow actually works

Everything above is a version of one of two shapes.

	Workflow	Agent
Who decides the steps?	Your code	The model, in a loop
Example	Triage: classify → validate → store	Tool loop: decide → call tool → observe → repeat
Predictability	High	Lower
Use when	Steps are known in advance	Steps depend on what's found along the way

Start with a workflow and only use an agent when you need the flexibility. Most useful production systems are workflows with one or two LLM steps inside them.

A typical real pipeline:

input → preprocess → [retrieve context] → LLM call → validate output
      → (tool calls if needed) → store result → log/trace → respond
================================================================================

In [ ]:
Stage 9: AI platform knowledge (what surrounds the code)

Once you can build the above, these are the concerns that separate a demo from a platform:

Evaluation: keep 20 to 50 test cases with expected outputs and run them on every prompt or model change. Without evals you're guessing whether things improved.
Observability: log prompts, responses, latency, token counts, and tool calls per request. Tools like Langfuse or OpenTelemetry help.
Cost and latency: track tokens per call, use prompt caching for repeated large prefixes, and use smaller models for simple steps.
Reliability: timeouts, retries with backoff, rate-limit handling, and fallbacks.
Guardrails and security: treat retrieved documents and user input as untrusted, because prompt injection is real. Give tools least privilege, and require confirmation for destructive actions.
Serving: wrap your logic in an HTTP API, for example with FastAPI:

python
                from fastapi import FastAPI
                app = FastAPI()

                @app.post("/ask")
                def ask_endpoint(q: dict):
                    return {"answer": ask(q["question"])}   # run: uvicorn main:app --reload
======================================================
Standards: MCP (Model Context Protocol) is how tools get packaged so any compatible AI app can use them.
It's worth learning once you're comfortable with tool calling.

Suggested 4-week plan
Week	Focus	Build
1	Stages 0 to 3	A script that pulls data from a public API, stores it in SQLite, and exports a CSV
2	Stages 4 to 5	The ticket triage tool, run over 30 sample tickets
3	Stages 6 to 7	A Q&A bot over your own documents, then add a tool call
4	Stages 8 to 9	Wrap it in FastAPI, add logging and an eval set of 20 questions